[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/01_openai_gym.ipynb)


# 01. Основы Gymnasium и случайный агент

Цель ноутбука — познакомиться с интерфейсом Gymnasium на среде CartPole-v1 и получить базовую линию качества для случайного агента.

**Результаты обучения:**
- создавать среду через `gym.make`;
- использовать современный API `reset` и `step`;
- отделять признаки завершения `terminated` и `truncated`;
- собирать статистику по эпизодам.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, главы 2-3.


In [ ]:
!pip install -q gymnasium


Подготовим библиотеки и зафиксируем генераторы случайных чисел. Блок с PyTorch оставлен общим для всех ноутбуков, даже если в этой части он не используется.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass


Создадим CartPole-v1 и посмотрим на пространства наблюдений и действий. Затем вручную сделаем несколько шагов, чтобы увидеть структуру кортежа, возвращаемого `step()`.


In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=SEED)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Sample observation:", obs)
print("Sample action:", env.action_space.sample())

for step_idx in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(step_idx, obs, reward, terminated, truncated, info)
    if terminated or truncated:
        obs, info = env.reset(seed=SEED + step_idx + 1)
env.close()


Теперь запустим два простых агента на 200 эпизодах: случайную базовую линию и детерминированную эвристику. Эвристика выбирает направление по углу шеста и угловой скорости, поэтому показывает, что задача решаема даже без обучения.


In [ ]:
def run_policy(policy_fn, episodes=200):
    env = gym.make("CartPole-v1")
    rewards, lengths = [], []
    for ep in range(episodes):
        obs, info = env.reset(seed=SEED + ep)
        total_reward, length = 0.0, 0
        done = False
        while not done:
            action = policy_fn(obs, env)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            length += 1
            done = terminated or truncated
        rewards.append(total_reward)
        lengths.append(length)
    env.close()
    return np.array(rewards), np.array(lengths)

random_rewards, random_lengths = run_policy(
    lambda obs, env: env.action_space.sample()
)
heuristic_rewards, heuristic_lengths = run_policy(
    lambda obs, env: int(obs[2] + 0.5 * obs[3] > 0)
)
print(f"Random mean reward: {random_rewards.mean():.2f}")
print(f"Heuristic mean reward: {heuristic_rewards.mean():.2f}")


Гистограмма сравнивает случайную политику и простую эвристику. Разрыв между ними удобен для отчёта: случайная линия остаётся честным baseline, а эвристика показывает достижимый максимум среды.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(random_rewards, bins=30, alpha=0.7, label="Random agent")
plt.hist(heuristic_rewards, bins=10, alpha=0.7, label="Heuristic agent")
plt.xlabel("Episode reward")
plt.ylabel("Count")
plt.title("CartPole baseline comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 2-3.


In [ ]:
os.makedirs("results", exist_ok=True)

def summarize(values):
    return [
        f"{np.mean(values):.2f}",
        f"{np.std(values):.2f}",
        f"{np.min(values):.0f}",
        f"{np.max(values):.0f}",
        f"{100 * np.mean(values >= 100):.1f} %",
        f"{100 * np.mean(values == 500):.1f} %",
    ]

metrics = [
    "Средняя суммарная награда за эпизод",
    "Стандартное отклонение награды",
    "Минимальная награда",
    "Максимальная награда",
    "Доля эпизодов с наградой ≥ 100",
    "Доля эпизодов с наградой = 500 (победа)",
]
df_results = pd.DataFrame({
    "Метрика": metrics,
    "Случайный агент": summarize(random_rewards),
    "Эвристический агент": summarize(heuristic_rewards),
})
print(df_results.to_string(index=False))
df_results.to_csv("results/01_random_baseline.csv", index=False)
